In [2]:
import pandas as pd
import gpxpy
import os

## Configuration
Remplacez par le chemin vers les fichiers à traiter

In [11]:
data_path = './input/AEHO007-22354823 2025-11-23 17_42_57 CET (Data CET).csv'
track_path = './input/Equipage Anita Conti.gpx'
instrument_name = 'AE_HO_007' # Suivre le format du numéro de série (owner_type_numéro)
output_dir = "output"

## Étape 1 : Charger les données GPX

In [12]:
def load_gpx(file_path):
    with open(file_path, 'r') as gpx_file:
        gpx = gpxpy.parse(gpx_file)
    points = []
    for track in gpx.tracks:
        for segment in track.segments:
            for point in segment.points:
                points.append({
                    'datetime': point.time,
                    'lat': point.latitude,
                    'long': point.longitude
                })
    
    df_gpx = pd.DataFrame(points)
    df_gpx['datetime'] = pd.to_datetime(df_gpx['datetime']).dt.tz_convert('UTC')  # Rendre naive
    return df_gpx

df_gpx = load_gpx(track_path)
df_gpx = df_gpx.set_index('datetime')

# Déterminer la plage de temps du GPX
gpx_start_at = df_gpx.index.min()
gpx_end_at = df_gpx.index.max()

print(f"Plage de temps du GPX : de {gpx_start_at} à {gpx_end_at}")

Plage de temps du GPX : de 2025-09-19 08:45:33+00:00 à 2025-09-20 15:36:13+00:00


## Étape 2 : Charger les données CSV

In [13]:
df = pd.read_csv(data_path, low_memory=False, parse_dates=[1], date_format="%m/%d/%Y %H:%M:%S")

# Renommer les colonnes
df = df.rename(columns={
    'Date et heure (CEST)': 'datetime',
    'Température   (°C)(W-CTD-00 22357933)': 'sea_temp',
    'Pression absolue   (kPa)(W-CTD-00 22357933)': 'air_pres',
    'Conductivité électrique   (µS/cm)(W-CTD-00 22357933)': 'ec_abs',
    'Conductivité spécifique   (µS/cm)(W-CTD-00 22357933)': 'ec25',
    'Salinité   (PSU)(W-CTD-00 22357933)': 'sea_sal',
    'Matières solides dissoutes totales   (mg/L)(W-CTD-00 22357933)': 'total_dissolved_solids',
})

# Supprimer les colonnes inutiles
df = df.drop(columns=['#','Hôte connecté', 'Fin du fichier', 'Démarré', 'Arrêté', 'Bouton Haut', 'Bouton Bas', 'Détection d’eau'])

# Convertir le fuseau horaire de CEST en UTC
df['datetime'] = df['datetime'].dt.tz_localize('Europe/Paris')
df['datetime'] = df['datetime'].dt.tz_convert('UTC')
df = df.set_index('datetime')

# Convertir la pression absolue de kPa en hPa
df['air_pres'] = df['air_pres'] * 10
df['air_pres'] = df['air_pres'].round(2)

In [14]:
# df = df[(df.name >= min_time_gpx) & (df.name <= max_time_gpx)]
print("Plage de dates CSV :", df.index.min(), "à", df.index.max())
print("Plage de dates CSV :", df.index.min(), "à", df.index.max())

Plage de dates CSV : 2025-07-21 16:43:27+00:00 à 2025-11-23 16:40:50+00:00
Plage de dates CSV : 2025-07-21 16:43:27+00:00 à 2025-11-23 16:40:50+00:00


## Étape 3 : Interpolation de la position des mesures à partir des traces GPS  

In [15]:
def interpolate_position(row, df_gpx):
    t = row.name
    # Trouver les deux points GPX encadrant t
    idx = df_gpx.index.searchsorted(t, side='right') - 1
    if idx < 0 or idx >= len(df_gpx) - 1:
        return None, None  # Hors des limites

    point_A = df_gpx.iloc[idx]
    point_B = df_gpx.iloc[idx + 1]

    tA, tB = point_A.name, point_B.name
    latA, longA = point_A['lat'], point_A['long']
    latB, longB = point_B['lat'], point_B['long']

    ratio = (t - tA).total_seconds() / (tB - tA).total_seconds()
    lat = latA + (latB - latA) * ratio
    long = longA + (longB - longA) * ratio

    return lat, long

In [16]:
df[['lat', 'long']] = df.apply(lambda row: interpolate_position(row, df_gpx), axis=1, result_type='expand')
df = df.dropna(subset=['lat', 'long']) # Nettoyer les valeurs manquantes

## Étape 4 : Exporter les données

In [ ]:
os.makedirs(output_dir, exist_ok=True)

start_at = df.index.min().strftime('%Y-%m-%d_%H:%M:%S')
output_csv_final = os.path.join(output_dir, f"{instrument_name}_{start_at}_with_positions.csv")
df.to_csv(output_csv_final, index=False)